In [4]:
from collections import deque
import heapq

class PuzzleState:
    def __init__(self, board, parent=None, action=None, cost=0):
        self.board = board  # Tupla, ex: (1, 2, 3, 4, 0, 5, 6, 7, 8)
        self.parent = parent
        self.action = action
        self.cost = cost

    def is_goal(self):
        return self.board == (1, 2, 3, 4, 5, 6, 7, 8, 0)

    def get_successors(self):
        successors = []
        zero_idx = self.board.index(0)
        row, col = divmod(zero_idx, 3)
        moves = {'Cima': (-1, 0), 'Baixo': (1, 0), 'Esquerda': (0, -1), 'Direita': (0, 1)}

        for action, (dr, dc) in moves.items():
            new_row, new_col = row + dr, col + dc
            if 0 <= new_row < 3 and 0 <= new_col < 3:
                new_idx = new_row * 3 + new_col
                new_board = list(self.board)
                new_board[zero_idx], new_board[new_idx] = new_board[new_idx], new_board[zero_idx]
                successors.append(
                    PuzzleState(tuple(new_board), parent=self, action=action, cost=self.cost + 1)
                )
        return successors

    def __lt__(self, other):
        # Necessário para heapq quando os custos f são iguais
        return self.cost < other.cost

print("PuzzleState definido com sucesso.")

PuzzleState definido com sucesso.


In [2]:
def bfs(start_state):
    """Busca em largura (BFS). Retorna (estado_objetivo, nós_expandidos)."""
    queue = deque([start_state])
    explored = set()
    explored.add(start_state.board)
    nodes_expanded = 0

    while queue:
        state = queue.popleft()
        nodes_expanded += 1

        if state.is_goal():
            return state, nodes_expanded

        for successor in state.get_successors():
            if successor.board not in explored:
                explored.add(successor.board)
                queue.append(successor)

    return None, nodes_expanded  # sem solução

print("bfs definido com sucesso.")

bfs definido com sucesso.


In [3]:
def manhattan_distance(board):
    """Soma das distâncias de Manhattan de cada peça até sua posição objetivo."""
    distance = 0
    for idx, tile in enumerate(board):
        if tile != 0:  # ignora o espaço vazio
            goal_row, goal_col = divmod(tile - 1, 3)   # posição objetivo (0-indexado)
            curr_row, curr_col = divmod(idx, 3)          # posição atual
            distance += abs(goal_row - curr_row) + abs(goal_col - curr_col)
    return distance


def a_star(start_state):
    """A* com heurística de Manhattan. Retorna (estado_objetivo, nós_expandidos)."""
    h0 = manhattan_distance(start_state.board)
    # Fila de prioridade: (f = g + h, estado)
    queue = [(h0, start_state)]
    explored = set()
    nodes_expanded = 0

    while queue:
        f, state = heapq.heappop(queue)

        if state.board in explored:
            continue  # já foi expandido com custo menor

        explored.add(state.board)
        nodes_expanded += 1

        if state.is_goal():
            return state, nodes_expanded

        for successor in state.get_successors():
            if successor.board not in explored:
                g = successor.cost
                h = manhattan_distance(successor.board)
                heapq.heappush(queue, (g + h, successor))

    return None, nodes_expanded  # sem solução

print("manhattan_distance e a_star definidos com sucesso.")

manhattan_distance e a_star definidos com sucesso.


In [8]:
import time

def get_path(state):
    """Reconstrói a sequência de ações do estado inicial até o objetivo."""
    path = []
    while state.parent:
        path.append(state.action)
        state = state.parent
    path.reverse()
    return path

# ──────────────────────────────────────────────
# Estado inicial — altere para testar diferentes dificuldades
# Fácil (~2 passos):   (1, 2, 3, 4, 0, 5, 7, 8, 6)
# Médio (~10 passos):  (1, 2, 3, 4, 5, 6, 0, 7, 8)
# Difícil (~20 passos):(5, 2, 8, 4, 1, 7, 0, 3, 6)
# ──────────────────────────────────────────────
initial_board = (5, 2, 8, 4, 1, 7, 0, 3, 6)
bfsnow = time.time()
start_bfs   = PuzzleState(initial_board)
start_astar = PuzzleState(initial_board)

bfs_time = time.time()
bfs_solution,   bfs_nodes   = bfs(start_bfs)
bfs_time = time.time() - bfs_time

astar_time = time.time()
astar_solution, astar_nodes = a_star(start_astar)
astar_time = time.time() - astar_time

bfs_path   = get_path(bfs_solution)
astar_path = get_path(astar_solution)

print("=" * 40)
print("               BFS")
print("=" * 40)
print(f"  Nós expandidos : {bfs_nodes}")
print(f"  Passos solução : {len(bfs_path)}")
print(f"  Tempo gasto     : {bfs_time:.4f} segundos")

print()
print("=" * 40)
print("               A*")
print("=" * 40)
print(f"  Nós expandidos : {astar_nodes}")
print(f"  Passos solução : {len(astar_path)}")
print(f"  Tempo gasto     : {astar_time:.4f} segundos")

print()
print("=" * 40)
print("           ANÁLISE COMPARATIVA")
print("=" * 40)
ratio = bfs_nodes / astar_nodes
print(f"  BFS expandiu {ratio:.1f}x mais nós que o A*!")
print(f"  Redução alcançada pelo A*: {(1 - astar_nodes/bfs_nodes)*100:.1f}% menos nós.")
print(f"  Tempo BFS: {bfs_time:.4f} s, Tempo A*: {astar_time:.4f} s")

               BFS
  Nós expandidos : 85698
  Passos solução : 22
  Tempo gasto     : 0.4145 segundos

               A*
  Nós expandidos : 672
  Passos solução : 22
  Tempo gasto     : 0.0055 segundos

           ANÁLISE COMPARATIVA
  BFS expandiu 127.5x mais nós que o A*!
  Redução alcançada pelo A*: 99.2% menos nós.
  Tempo BFS: 0.4145 s, Tempo A*: 0.0055 s
